# Membrane

In [ ]:
import itertools
import logging
import os
import random
from copy import deepcopy
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from shapely import to_wkt, wkt
from shapely.affinity import rotate, translate
from shapely.geometry import MultiPolygon, Point, Polygon, box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from tqdm import tqdm

import cyanomembranes as cm

(OUT := Path("output")).mkdir(exist_ok=True)

# Random generator
RNG = np.random.default_rng(seed=0)

# CPUS
N_PROCESSES = 10 #6

# Create protein shadows

In [ ]:
PDB_FILES = [
    "3JCU-PSII-spinach.pdb",
    "6RQF-Cyt-spinach.pdb",
    "7E0K-LHCII-chlamy.pdb"
]

DATA = Path("data_plants/")

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)


In [ ]:
def plot_membrane(filepath, ax):
    p = cm.geo_utils.readwkt(filepath)
    psi_area = proteins["3JCU-PSII-spinach"]["polygon"][0].area
    psii_area = proteins["6RQF-Cyt-spinach"]["polygon"][0].area
    cytb6f_area = proteins["7E0K-LHCII-chlamy"]["polygon"][0].area

    pastel_blue = "#A2CFFE"
    pastel_yellow = "#F7E1B5"
    pastel_green = "#A9E5B8"
    pastel_darkblue = "#54A5FC"

    psii_lst = []
    for i in p:
        if np.isclose(i.area, psi_area):
            ax.fill(*i.exterior.xy, lw=0.4, c=pastel_blue, edgecolor="grey")
        if np.isclose(i.area, psii_area):
            ax.fill(*i.exterior.xy, lw=0.4, c=pastel_yellow, edgecolor="grey")
            psii_lst.append(i)
        if np.isclose(i.area, cytb6f_area):
            ax.fill(*i.exterior.xy, lw=0.4, c="red", edgecolor="grey")



    ax.set_aspect("equal")
    ax.set_xlim(0,5000)
    ax.set_ylim(0,5000)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 8))
ax = ax.flatten()
for i, prot in enumerate(proteins):
    polygon = proteins[prot]["polygon"]
    ax[i].plot(*polygon[0].exterior.xy)
    ax[i].set_aspect("equal")
    ax[i].set_title(prot)
    ax[i].set_xlim(-150, 150)
    ax[i].set_ylim(-150, 150)
plt.tight_layout()
fig.show()

## Create membranes

In [ ]:
# ----------------------------------

# CONFIGURATION

# ----------------------------------



NUMBER_OF_PROTEINS = {
    "avg_membrane": [10, 40, 80, 100, 110, 117]
}

ANGLES = range(360)
N = 10

# -----------------------------------

# LOGGING

# -----------------------------------

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO
)

# --------------------------------

# UTILITY FUNCTIONS

# --------------------------------


def ensure_output_dir(out_dir: Path) -> None:
    """Ensure output directory exists"""
    out_dir.mkdir(parents=True, exist_ok=True)


def generate_rotated_polygons(
    n: int, base_polygon: Polygon, angles: range
) -> list[Polygon]:
    """Generate n rotated copies of a base polygon"""
    return [rotate(base_polygon, random.choice(angles)) for _ in range(n)]


def generate_roated_polygons_membrane(n_cytb6f: int, angles: range) -> list[Polygon]:
    return (
        [
            rotate(proteins["3JCU-PSII-spinach"]["polygon"][0], random.choice(ANGLES))
            for _ in range(int(n_cytb6f * 2.6))
        ]
        + [
            rotate(
                proteins["6RQF-Cyt-spinach"]["polygon"][0],
                random.choice(ANGLES),
            )
            for _ in range(int(n_cytb6f))
        ]
        + [
            rotate(proteins["7E0K-LHCII-chlamy"]["polygon"][0], random.choice(ANGLES))
            for _ in range(int(n_cytb6f * 14.1))
        ]
    )


def save_polygons_to_wkt(file_path: Path, polygons: list[Polygon]) -> None:
    """Save polygons as WKT to a file"""
    with file_path.open("w") as f:
        for poly in polygons:
            f.write(poly.wkt + "\n")


# --------------------------------

# MAIN WORKER FUNCTION

# --------------------------------


def process_one_case(args: tuple[str, int, int, str]) -> Path:
    polygon_key, nprot, i = args

    # Polygon source and output directory
    if polygon_key != "avg_membrane":
        base_polygon = proteins[polygon_key]["polygon"][0]
    out_dir = OUT / polygon_key
    ensure_output_dir(out_dir)

    file_path = Path(out_dir / f"polygons_{polygon_key}_{nprot}-{i}.wkt")

    if file_path.exists():
        logging.info(f"{file_path.name} already exists, skipping.")
        return file_path

    if polygon_key == "avg_membrane":
        polygons = generate_roated_polygons_membrane(nprot, ANGLES)
    else:
        polygons = generate_rotated_polygons(nprot, base_polygon, ANGLES)
    world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
        polygons,
        dimensions=[5000, 5000],
        step_size=15,
        rotation_angle=360,
        max_iter_placement=10000,
    )

    extended_world = world + list(np.array(ghosts).ravel())
    save_polygons_to_wkt(file_path, extended_world)
    logging.info(f"Saved {len(extended_world)} polygons to {file_path}")

    return file_path


tasks = [
    (polygon_key, nprot, i)
    for polygon_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for i in range(N)
]

with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)

In [ ]:
fig, ax = plt.subplots()
plot_membrane("output/avg_membrane/polygons_avg_membrane_117-0.wkt", ax=ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
plot_membrane("output/avg_membrane/polygons_avg_membrane_110-0.wkt", ax=ax)
fig.show()

In [ ]:
p117 = cm.geo_utils.readwkt("output/avg_membrane/polygons_avg_membrane_117-0.wkt")
rec = box(0, 0, 5000, 5000)
inter = rec.intersection(unary_union(p117))
inter.area/ rec.area

In [ ]:
p110 = cm.geo_utils.readwkt("output/avg_membrane/polygons_avg_membrane_110-0.wkt")
rec = box(0, 0, 5000, 5000)
inter = rec.intersection(unary_union(p110))
inter.area/ rec.area